# Data Exploration - Phase 1

This notebook demonstrates the data ingestion layer and explores market + news data.

## Goals
1. Fetch market data for the ticker universe
2. Fetch news data for sample tickers
3. Save data to parquet files
4. Basic exploration and visualization
5. Identify any obvious price drops to validate the data pipeline

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date, timedelta
from pathlib import Path
import yaml
import logging
from dotenv import load_dotenv

# Load environment variables (for NEWSAPI_AI_KEY)
load_dotenv()

# Configure logging
logging.basicConfig(level=logging.INFO)

# Project imports
from src.ingestion.market.yahoo_market import YahooMarketData
from src.ingestion.news.newsapi_source import NewsAPISource

print("Imports successful!")

## 1. Load Universe Configuration

In [ ]:
# Load ticker universe from config
with open('../config/tickers.yaml', 'r') as f:
    tickers_config = yaml.safe_load(f)

# Extract all tickers from universe
universe = []
for sector, stocks in tickers_config['universe'].items():
    for stock in stocks:
        universe.append({
            'symbol': stock['symbol'],
            'name': stock['name'],
            'sector': sector,
            'sector_etf': stock.get('sector_etf')
        })

universe_df = pd.DataFrame(universe)
print(f"Universe contains {len(universe_df)} stocks")
display(universe_df)

In [ ]:
# Extract benchmark tickers
benchmarks = []
for category, items in tickers_config['benchmarks'].items():
    for item in items:
        benchmarks.append({
            'symbol': item['symbol'],
            'name': item['name'],
            'category': category
        })

benchmarks_df = pd.DataFrame(benchmarks)
print(f"Benchmarks: {len(benchmarks_df)} symbols")
display(benchmarks_df)

## 2. Fetch Market Data

In [ ]:
# Initialize market data provider
market_data = YahooMarketData()

# Define date range (1 year of data)
end_date = date.today()
start_date = end_date - timedelta(days=365)

print(f"Fetching data from {start_date} to {end_date}")

In [ ]:
# Test with a single ticker first (AAPL)
test_ticker = 'AAPL'
aapl_data = market_data.fetch_ohlcv(test_ticker, start_date, end_date)

print(f"Fetched {len(aapl_data.bars)} bars for {test_ticker}")
print(f"Metadata: {aapl_data.metadata}")

# Convert to DataFrame and display
aapl_df = aapl_data.to_dataframe()
print(f"\nFirst 5 rows:")
display(aapl_df.head())
print(f"\nLast 5 rows:")
display(aapl_df.tail())

In [ ]:
# Fetch all universe tickers + key benchmarks
all_tickers = universe_df['symbol'].tolist()
benchmark_tickers = ['SPY', 'QQQ']  # Main benchmarks
sector_etfs = ['XLK', 'XLV', 'XLF', 'XLP', 'XLI', 'XLU']

fetch_tickers = all_tickers + benchmark_tickers + sector_etfs
print(f"Fetching {len(fetch_tickers)} tickers...")

# Batch fetch
all_data = market_data.fetch_batch(fetch_tickers, start_date, end_date)

# Show fetch results
for ticker, data in all_data.items():
    print(f"{ticker}: {len(data.bars)} bars")

In [ ]:
# Combine all data into a single DataFrame for easier analysis
combined_dfs = []
for ticker, data in all_data.items():
    if data.bars:
        df = data.to_dataframe()
        df['ticker'] = ticker
        combined_dfs.append(df.reset_index())

market_df = pd.concat(combined_dfs, ignore_index=True)
print(f"Combined market data: {len(market_df)} rows")
display(market_df.head())

## 3. Save Market Data

In [ ]:
# Create data directory if it doesn't exist
data_dir = Path('../data/raw/market')
data_dir.mkdir(parents=True, exist_ok=True)

# Save combined data as parquet
output_file = data_dir / f'market_data_{start_date}_{end_date}.parquet'
market_df.to_parquet(output_file, index=False)
print(f"Saved market data to {output_file}")

# Show file size
import os
file_size = os.path.getsize(output_file) / (1024 * 1024)
print(f"File size: {file_size:.2f} MB")

## 4. Fetch News Data

In [ ]:
# Initialize news source (requires NEWSAPI_AI_KEY environment variable)
news_source = NewsAPISource()

# Test with a single ticker
test_ticker = 'AAPL'
aapl_news = news_source.fetch_for_ticker(test_ticker, lookback_hours=168)  # 1 week

print(f"Found {len(aapl_news)} news articles for {test_ticker}")
for article in aapl_news[:5]:
    print(f"\n- {article.title}")
    print(f"  Source: {article.source}")
    print(f"  Published: {article.published_at}")
    print(f"  URL: {article.url}")
    if article.content:
        print(f"  Content preview: {article.content[:200]}...")

In [ ]:
# Fetch news for all universe tickers (this may take a while due to rate limiting)
all_news = []
sample_tickers = universe_df['symbol'].tolist()[:6]  # Start with first 6 for testing

print(f"Fetching news for {len(sample_tickers)} tickers (sample)...")
for ticker in sample_tickers:
    articles = news_source.fetch_for_ticker(ticker, lookback_hours=168)
    for article in articles:
        article_dict = article.to_dict()
        article_dict['primary_ticker'] = ticker
        all_news.append(article_dict)
    print(f"{ticker}: {len(articles)} articles")

news_df = pd.DataFrame(all_news)
print(f"\nTotal news articles: {len(news_df)}")
display(news_df.head())

In [ ]:
# Save news data
news_dir = Path('../data/raw/news')
news_dir.mkdir(parents=True, exist_ok=True)

news_file = news_dir / f'news_data_{date.today()}.parquet'
if len(news_df) > 0:
    news_df.to_parquet(news_file, index=False)
    print(f"Saved news data to {news_file}")
else:
    print("No news data to save")

## 5. Basic Price Analysis - Find Drops

In [ ]:
# Calculate daily returns for each ticker
market_df['date'] = pd.to_datetime(market_df['timestamp']).dt.date
market_df = market_df.sort_values(['ticker', 'timestamp'])

# Calculate daily return (close-to-close)
market_df['prev_close'] = market_df.groupby('ticker')['close'].shift(1)
market_df['daily_return'] = (market_df['close'] - market_df['prev_close']) / market_df['prev_close']

display(market_df[['ticker', 'timestamp', 'close', 'daily_return']].head(10))

In [ ]:
# Find significant drops (> 2%)
drop_threshold = -0.02
drops = market_df[market_df['daily_return'] < drop_threshold].copy()
drops = drops.sort_values('daily_return')

print(f"Found {len(drops)} daily drops > 2% in the dataset")
print(f"\nTop 20 largest drops:")
display(drops[['ticker', 'timestamp', 'close', 'daily_return']].head(20))

In [ ]:
# Drops by sector
drops_with_sector = drops.merge(
    universe_df[['symbol', 'sector']], 
    left_on='ticker', 
    right_on='symbol', 
    how='left'
)

sector_drops = drops_with_sector.groupby('sector').agg({
    'daily_return': ['count', 'mean', 'min']
}).round(4)
sector_drops.columns = ['count', 'avg_drop', 'max_drop']
print("Drops by sector:")
display(sector_drops.sort_values('count', ascending=False))

## 6. Visualization

In [ ]:
# Plot price history for a sample stock with drops highlighted
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sample_ticker = 'AAPL'
ticker_data = market_df[market_df['ticker'] == sample_ticker].copy()
ticker_drops = ticker_data[ticker_data['daily_return'] < drop_threshold]

# Price chart
ax1 = axes[0]
ax1.plot(ticker_data['timestamp'], ticker_data['close'], label='Close Price')
ax1.scatter(ticker_drops['timestamp'], ticker_drops['close'], 
           color='red', s=50, label=f'Drops > {abs(drop_threshold)*100}%', zorder=5)
ax1.set_title(f'{sample_ticker} Price History')
ax1.set_ylabel('Price ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Daily returns
ax2 = axes[1]
colors = ['red' if r < drop_threshold else 'green' if r > 0 else 'gray' 
          for r in ticker_data['daily_return']]
ax2.bar(ticker_data['timestamp'], ticker_data['daily_return'] * 100, color=colors, alpha=0.7)
ax2.axhline(y=drop_threshold * 100, color='red', linestyle='--', label=f'{drop_threshold*100}% threshold')
ax2.set_title(f'{sample_ticker} Daily Returns')
ax2.set_ylabel('Return (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily returns
fig, ax = plt.subplots(figsize=(10, 6))

returns = market_df['daily_return'].dropna()
ax.hist(returns * 100, bins=100, edgecolor='black', alpha=0.7)
ax.axvline(x=drop_threshold * 100, color='red', linestyle='--', 
          label=f'Drop threshold ({drop_threshold*100}%)')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Daily Returns (All Universe Stocks)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Return statistics:")
print(f"  Mean: {returns.mean()*100:.2f}%")
print(f"  Std:  {returns.std()*100:.2f}%")
print(f"  Min:  {returns.min()*100:.2f}%")
print(f"  Max:  {returns.max()*100:.2f}%")

## 7. Summary

This notebook demonstrated:
1. Loading ticker universe from config
2. Fetching 1 year of market data for all universe stocks + benchmarks
3. Fetching news for sample tickers
4. Saving data to parquet files
5. Basic drop detection and visualization

Next steps:
- Implement market filtering (beta-adjusted returns)
- Entity resolution for news
- Correlate drops with news events

In [ ]:
# Summary stats
print("=" * 50)
print("DATA EXPLORATION SUMMARY")
print("=" * 50)
print(f"Date range: {start_date} to {end_date}")
print(f"Universe stocks: {len(universe_df)}")
print(f"Total market data rows: {len(market_df)}")
print(f"Significant drops (>{abs(drop_threshold)*100}%): {len(drops)}")
if len(news_df) > 0:
    print(f"News articles collected: {len(news_df)}")
print("=" * 50)